In [95]:
import os
import sys
sys.path.append('/workspace/Retinal-vessels-segmentation/')

In [96]:
import numpy as np
from PIL import Image

In [97]:
import torch
import torch.nn as nn
from utils import *
import matplotlib.pyplot as plt
import cv2
from io import BytesIO
import torch.nn.functional as F
from transforms import get_test_patch_transforms
from sklearn.metrics import f1_score, recall_score

In [ ]:
output_dir='./output'

In [ ]:
def draw_dashed_line(img, pt1, pt2, color, thickness=1, gap=10):
    """Draw a dashed line between two points"""
    dist = ((pt1[0] - pt2[0]) ** 2 + (pt1[1] - pt2[1]) ** 2) ** 0.5
    pts = []
    for i in np.arange(0, dist, gap):
        r = i / dist
        x = int((pt1[0] * (1 - r) + pt2[0] * r) + 0.5)
        y = int((pt1[1] * (1 - r) + pt2[1] * r) + 0.5)
        pts.append((x, y))
    
    for i in range(0, len(pts) - 1, 2):
        cv2.line(img, pts[i], pts[i + 1], color, thickness)

In [ ]:
def save_png_to_new_path(input_path, root_output_path,suf=''):
    img_name = input_path.split('/')[-1].split('.')[0]
    output_path = os.path.join(root_output_path, f'{img_name}_{suf}.png')
    img = cv2.imread(input_path,1)
    cv2.imwrite(output_path, img)

In [ ]:
def swap_loc(imgs):
    def filter_key(path):
        if '_gt.' in path:
            return False
        elif '_origin.' in path:
            return False
        elif 'our_net' in path:
            return False
        else :
            pass
        return True
    tmp = [0]*3
    for path in imgs:
        if '_gt.' in path:
            tmp[1] = path
        elif '_origin.' in path:
            tmp[0] = path
        elif 'our_net' in path:
            tmp[2] = path
        else : pass
    return tmp+list(filter(filter_key, imgs))

In [ ]:
import glob
class  Drawer:
    def __init__(self, models=['dysta_net','edae_net','fr_net','gtdla','our_net','sfit_net','unet']
                 ,checkpoints_path='/workspace/Retinal-vessels-segmentation/checkpoints',
                 output_dir='./output',
                 resize_size=(1024,1024)):
        self.models = models
        self.output_dir = output_dir
        self.checkpoints_path = checkpoints_path
        self.resize_size = resize_size
        if self.output_dir and not os.path.exists(self.output_dir):
            os.makedirs(self.output_dir, exist_ok=True)
    def infer_model(self,model, image,preprocessing_func,get_test_patch_transforms=get_test_patch_transforms):
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = model.to(device)
        model.eval()
        with torch.inference_mode():
            if isinstance(image, str):
                image = np.array(Image.open(image))
            preprocessed_image = preprocessing_func(image)
            img_tensor = get_test_patch_transforms()(image=preprocessed_image)['image'].to(device)
            _, original_h, original_w = img_tensor.shape

            
            img_tensor = mirror_padding_v2(img_tensor).unsqueeze(0)
            B, C, H, W = img_tensor.shape
            num_patch = ((H-64)//32+1, (W-64)//8+1)
            image_patches, tmp_stride = extract_patches_with_target_count(img_tensor, 64, num_patch)
            if len(image_patches.shape) > 4:
                image_patches = image_patches.flatten(0, 1)
            chunk_size = max(image_patches.shape[0] // 128, 1)
            chunk_image = torch.chunk(image_patches, chunk_size, 0)

            out_sample = []
            for c_image in chunk_image:
                with torch.inference_mode():
                    prob = model(c_image)
                out_sample.append(prob)
            prob = torch.cat(out_sample, 0)
            prob = prob.view(B, -1, 1, 64, 64)
            prob = reverse_to_original_image(prob, (H, W), 64, tmp_stride).squeeze()[:original_h, :original_w]

            # Threshold => binary mask (numpy)
            pred_mask = (prob >= 0.487).to(torch.uint8).detach().cpu().numpy()  # shape (h,w), 0/1
            display_mask = (pred_mask * 255).astype(np.uint8)
        return display_mask
    def create_zoom_inset(self,path,out_path,pos=(243, 419, 48, 30),resize_size=(1024,1024), line_style='dashed'):
        """
        line_style options:
        - 'solid': solid lines
        - 'dashed': dashed lines
        - 'both': connects all 4 corners
        - 'diagonal': connects diagonal corners
        """

        img = cv2.imread(path,1)
        h, w = img.shape[:2]

        # 1. Xác định vị trí vùng muốn cắt (x, y, width, height)
        crop_x, crop_y, crop_w, crop_h = pos 
        
        # 2. Cắt vùng đó ra
        roi = img[crop_y:crop_y+crop_h, crop_x:crop_x+crop_w]
        
        # 3. Phóng to vùng đã cắt
        zoom_scale = 20
        zoomed_roi = cv2.resize(roi, None, fx=zoom_scale, fy=zoom_scale, interpolation=cv2.INTER_LANCZOS4)

        # 4. Vẽ khung vàng cho vùng cắt trên ảnh gốc
        cv2.rectangle(img, (crop_x, crop_y), (crop_x+crop_w, crop_y+crop_h), (0, 255, 0), 2)

        # 5. Xác định vị trí đặt ảnh đã zoom
        zh, zw = zoomed_roi.shape[:2]
        pos_x, pos_y = w - zw, h - zh
        
        # Vẽ khung vàng cho ảnh zoom
        cv2.rectangle(zoomed_roi, (0, 0), (zw-1, zh-1), (0, 255, 0), 8)

        # 6. Đè ảnh zoom lên ảnh gốc
        img[pos_y:pos_y+zh, pos_x:pos_x+zw] = zoomed_roi

        # 7. VẼ CÁC ĐƯỜNG NỐI
        line_color = (0, 255, 255)  # Yellow (BGR)
        line_thickness = 5
        
        # Define corner points of ROI (original region)
        roi_corners = {
            'top_left': (crop_x, crop_y),
            'top_right': (crop_x + crop_w, crop_y),
            'bottom_left': (crop_x, crop_y + crop_h),
            'bottom_right': (crop_x + crop_w, crop_y + crop_h)
        }
        
        # Define corner points of zoomed inset
        zoom_corners = {
            'top_left': (pos_x, pos_y),
            'top_right': (pos_x + zw, pos_y),
            'bottom_left': (pos_x, pos_y + zh),
            'bottom_right': (pos_x + zw, pos_y + zh)
        }
        
        if line_style == 'solid':
            # Connect corresponding corners with solid lines
            cv2.line(img, roi_corners['top_right'], zoom_corners['top_left'], line_color, line_thickness)
            cv2.line(img, roi_corners['bottom_right'], zoom_corners['bottom_left'], line_color, line_thickness)
        
        elif line_style == 'dashed':
            # Connect corresponding corners with dashed lines
            draw_dashed_line(img, roi_corners['top_right'], zoom_corners['top_left'], line_color, line_thickness)
            draw_dashed_line(img, roi_corners['bottom_right'], zoom_corners['bottom_left'], line_color, line_thickness)
        
        elif line_style == 'both':
            # Connect all 4 corners
            draw_dashed_line(img, roi_corners['top_left'], zoom_corners['top_left'], line_color, line_thickness)
            draw_dashed_line(img, roi_corners['top_right'], zoom_corners['top_right'], line_color, line_thickness)
            draw_dashed_line(img, roi_corners['bottom_left'], zoom_corners['bottom_left'], line_color, line_thickness)
            draw_dashed_line(img, roi_corners['bottom_right'], zoom_corners['bottom_right'], line_color, line_thickness)
        
        elif line_style == 'diagonal':
            # Connect diagonal corners
            draw_dashed_line(img, roi_corners['top_left'], zoom_corners['bottom_right'], line_color, line_thickness)
            draw_dashed_line(img, roi_corners['bottom_right'], zoom_corners['top_left'], line_color, line_thickness)
        img = cv2.resize(img, resize_size, interpolation=cv2.INTER_LANCZOS4)
        cv2.imwrite(out_path, img)
        roi_path = '/'.join(out_path.split('/')[:-2]+['roi']+[out_path.split('/')[-1]])
        os.makedirs('/'.join(roi_path.split('/')[:-1]), exist_ok=True)

        zoom_roi_path = '/'.join(out_path.split('/')[:-2]+['zoom_roi']+[out_path.split('/')[-1]])
        os.makedirs('/'.join(zoom_roi_path.split('/')[:-1]), exist_ok=True)

        cv2.imwrite(roi_path, roi)
        cv2.imwrite(zoom_roi_path, zoomed_roi)
    def concatenate_images_simple(self,images, padding=1, bg_color=(0, 0, 0),type_cat='vertical'):
        images = [np.array(cv2.imread(images[i],1)) for i in range(len(images))]
        
        if padding > 0:
            h, w, c = images[0].shape
            padding_strip = np.full((h, padding, c), bg_color, dtype=np.uint8)
            
            result = []
            for i, img in enumerate(images):
                result.append(img)
                if i < len(images) - 1:
                    result.append(padding_strip)
            if type_cat=='vertical':
                return np.vstack(result)
            elif type_cat=='horizontal':
                return np.hstack(result)
        else:
            if type_cat=='vertical':
                return np.vstack(images)
            elif type_cat=='horizontal':
                return np.hstack(images)
    def run(self, input_path,gt_path,suffix='drive',only_vs=True,pos=None,ablation=False):
        if suffix == '':
            suffix_dir = 'ablation'
        else:
            suffix_dir = suffix
        os.makedirs(self.output_dir, exist_ok=True)
        os.makedirs(os.path.join(self.output_dir,suffix_dir), exist_ok=True)
        os.makedirs(os.path.join(self.output_dir,suffix_dir,'zoom_inset'), exist_ok=True)
        os.makedirs(os.path.join(self.output_dir,suffix_dir,'infer'), exist_ok=True)
        os.makedirs(os.path.join(self.output_dir,suffix_dir,'cat'), exist_ok=True)
        os.makedirs(os.path.join(self.output_dir,suffix_dir,'pos'), exist_ok=True)
        save_png_to_new_path(input_path, os.path.join(self.output_dir,suffix_dir,'infer'),suf='origin')
        save_png_to_new_path(gt_path, os.path.join(self.output_dir,suffix_dir,'infer'),suf='gt')
        gt = np.ceil(cv2.imread(gt_path,0).astype(np.float32)/255).astype(np.uint8)
        recalls= []
        f1_scores = []
        tmp_paths = []
        for model_name in self.models:
            # print(input_path.split("/")[-1].split(".")[0])
            print(model_name)
            from load_model import load_model_class
            load_model_class(model_name if model_name != 'ours_woLoss' else 'our_net')
            model = torch.load(
                    os.path.join(self.checkpoints_path, f'{model_name}_{suffix}.pt') if suffix!='' else os.path.join(self.checkpoints_path, f'{model_name}.pt'),
                    map_location='cuda' if torch.cuda.is_available() else 'cpu',
                    weights_only=False
            )
            dis = self.infer_model(model, input_path,preprocessing_img if  model_name != 'our_net' else lambda x: cv2.cvtColor(x, cv2.COLOR_RGB2GRAY),
                                   get_test_patch_transforms_v1 if (ablation and model_name!='our_net') else get_test_patch_transforms)
            dis_0_1 = np.ceil(dis.astype(np.float32)/255).astype(np.uint8)
            recall = recall_score(gt.flatten(), dis_0_1.flatten())
            f1 = f1_score(gt.flatten(), dis_0_1.flatten())
            recalls.append(recall)
            f1_scores.append(f1)
            save_path = os.path.join(self.output_dir,suffix_dir,'infer',f'{input_path.split("/")[-1].split(".")[0]}_{model_name}.png')
            tmp_paths.append(save_path)
            cv2.imwrite(save_path, dis)
        imgs = glob.glob(os.path.join(self.output_dir,suffix_dir,'infer',f'{input_path.split("/")[-1].split(".")[0]}*.png')) + glob.glob(os.path.join(self.output_dir,suffix_dir,'infer',f'*_gt.png'))
        imgs = swap_loc(imgs)
        tmp_paths=imgs[:2]+tmp_paths
        if only_vs:
            fig,ax = plt.subplots(1,len(imgs),figsize=(10,5))
            our_net_path = imgs[2]
            our_net_img = np.ceil(cv2.imread(our_net_path,0).astype(np.float32)/255).astype(np.uint8)
            intersec=our_net_img*gt
            fn_if = np.ones_like(gt)
            for i in range(3,len(imgs)):
                if_img = np.ceil(cv2.imread(imgs[i],0).astype(np.float32)/255).astype(np.uint8)
                intersec_dis = (np.clip(intersec - (if_img*gt),0,1)*255).astype(np.uint8)
                fn_if*=(intersec_dis//255).astype(np.uint8)
                ax[i-3].imshow(intersec_dis,cmap='gray')
                ax[i-3].set_title(f'Our net vs {imgs[i].split("_")[-2].split(".")[0]}')
                ax[i-3].axis('off')
            fn_if = np.full((fn_if.shape[0],fn_if.shape[1],3),1)*fn_if.reshape((fn_if.shape[0],fn_if.shape[1],1))*np.array([0,0,255]).astype(np.uint8)
            cv2.imwrite(os.path.join(self.output_dir,suffix_dir,'pos',f'{input_path.split("/")[-1].split(".")[0]}_fn_if.png'),fn_if)
            ax[-3].imshow(fn_if)
            ax[-3].set_title(f'lack of intersection')
            ax[-3].axis('off')
            ax[-2].imshow(our_net_img,cmap='gray')
            ax[-2].set_title(f'pred')
            ax[-2].axis('off')
            ax[-1].imshow(Image.open(os.path.join(self.output_dir,suffix_dir,'infer',f'{input_path.split("/")[-1].split(".")[0]}_origin.png')))
            ax[-1].set_title(f'original')
            ax[-1].axis('off')
            plt.tight_layout()
            plt.show()
            print('Recalls:', self.models[np.argmax(recalls)])
            print('F1-scores:', self.models[np.argmax(f1_scores)])
        else:
            if pos is not None:
                for path in imgs if ablation==False else tmp_paths:
                    print(path)
                    self.create_zoom_inset(path,os.path.join(self.output_dir,suffix_dir,'zoom_inset', 'zoom_'+path.split("/")[-1]),pos=pos,line_style='dashed')
                cat_h_img = self.concatenate_images_simple(glob.glob(os.path.join(self.output_dir,suffix_dir,'zoom_inset','*.png')), padding=0, bg_color=(0, 0, 0),type_cat='horizontal')
                cat_v_img = self.concatenate_images_simple(glob.glob(os.path.join(self.output_dir,suffix_dir,'zoom_inset','*.png')), padding=0, bg_color=(0, 0, 0),type_cat='vertical')
                save_path_zoom = os.path.join(self.output_dir,suffix_dir,'cat','h_cat.png')
                save_path_zoom_2 = os.path.join(self.output_dir,suffix_dir,'cat','v_cat.png')
                cv2.imwrite(save_path_zoom, cat_h_img)
                Image.open(save_path_zoom).save(
                    save_path_zoom,
                    dpi=(300, 300)
                )
                cv2.imwrite(save_path_zoom_2, cat_v_img)
                Image.open(save_path_zoom_2).save(
                    save_path_zoom_2,
                    dpi=(300, 300)
                )
        # else:

In [ ]:
drawer = Drawer(models=['dysta_net','edae_net','fr_net','gtdla','our_net','sfit_net','unet'],
                 checkpoints_path='/workspace/Retinal-vessels-segmentation/checkpoints',
                 output_dir='./output',
                 resize_size=(1024,1024))


In [ ]:
drawer_ab = Drawer(models=['baseline','baseline_RHMA_woMDSA','baseline_RHMA_SAG_woMDSA','baseline_RHMA_SAG_MDSA','ours_woLoss','our_net'],
                 checkpoints_path='/workspace/Retinal-vessels-segmentation/checkpoints',
                 output_dir='./output',
                 resize_size=(1024,1024))


In [ ]:
img_paths= sorted(glob.glob('/workspace/Retinal-vessels-segmentation/data/CHASEDB_1/*/images/*'))
gt_paths = sorted(glob.glob('/workspace/Retinal-vessels-segmentation/data/CHASEDB_1/*/mask/*'))

In [ ]:
for i in range(len(img_paths)): # loop to find image have highest recall and highest f1 in ablation study
    drawer_ab.run(img_paths[i],gt_paths[i],suffix='',only_vs=True,ablation=True)

In [ ]:
for i in range(len(img_paths)): # loop to find image have highest recall and highest f1 in normal method
    print(img_paths[i].split('/')[-1])
    drawer.run(img_paths[i],gt_paths[i],suffix='chasedb',only_vs=True,ablation=False)

In [ ]:
drawer.run('/workspace/Retinal-vessels-segmentation/data/DRIVE/test/images/03_test.tif',
           '/workspace/Retinal-vessels-segmentation/data/DRIVE/test/mask/03_manual1.gif',
           suffix='drive',
           only_vs=True,)

In [ ]:
drawer.run('/workspace/Retinal-vessels-segmentation/data/DRIVE/test/images/03_test.tif',
           '/workspace/Retinal-vessels-segmentation/data/DRIVE/test/mask/03_manual1.gif',
           suffix='drive',
           only_vs=True,)

In [ ]:
def score_image_for_visualization(gt_bin, pred_bins):
    """
    Score how well an image demonstrates progressive improvement.
    Higher = better for visualization.
    Returns: (total_score, details_dict)
    """
    n = len(pred_bins)
    step_gains = []
    step_regressions = []

    for i in range(1, n):
        prev_fn = gt_bin * (1 - pred_bins[i - 1])
        recovered = prev_fn * pred_bins[i] * gt_bin
        regression = (pred_bins[i - 1] * gt_bin) * (1 - pred_bins[i])
        step_gains.append(recovered.sum())
        step_regressions.append(regression.sum())

    step_gains = np.array(step_gains, dtype=np.float32)
    step_regressions = np.array(step_regressions, dtype=np.float32)

    # 1. Total gain (more recovered pixels = better)
    total_gain = step_gains.sum()

    # 2. Monotonicity: all steps should have net positive gain
    net_gains = step_gains - step_regressions
    mono_ratio = (net_gains > 0).sum() / len(net_gains)  # 1.0 = perfect

    # 3. Evenness: gains spread across steps, not just one big jump
    if total_gain > 0:
        proportions = step_gains / total_gain
        # Entropy-like: uniform distribution = best
        evenness = -np.sum(proportions * np.log(proportions + 1e-8))
        max_entropy = np.log(len(step_gains))
        evenness_ratio = evenness / max_entropy if max_entropy > 0 else 0
    else:
        evenness_ratio = 0

    # 4. Low regression penalty
    reg_penalty = step_regressions.sum() / (total_gain + 1)

    score = total_gain * mono_ratio * (0.5 + 0.5 * evenness_ratio) / (1 + reg_penalty)

    details = {
        'step_gains': step_gains.tolist(),
        'step_regressions': step_regressions.tolist(),
        'mono_ratio': mono_ratio,
        'evenness': evenness_ratio,
        'reg_penalty': reg_penalty,
        'score': score
    }
    return score, details

In [ ]:
"""
Ablation Study Visualization
- Overlay: Green=TP, Red=FP, Blue=FN
- Auto-find 2 best ROI regions (progressive improvement)
- Extract: full image with ROI boxes + zoomed ROI crops
- Output structure per image per model: full_overlay.png, zoom_0.png, zoom_1.png
"""

import os
import cv2
import glob
import numpy as np
from PIL import Image
from scipy.signal import fftconvolve


# ============================================================
# 1. COLOR OVERLAY (TP=Green, FP=Red, FN=Blue)
# ============================================================
def create_tp_fp_fn_overlay(pred_binary, gt_binary, bg_image=None):
    """
    pred_binary, gt_binary: (H,W) uint8, values 0 or 1
    bg_image: optional (H,W,3) BGR background; if None, use black
    Returns: (H,W,3) BGR overlay image
    """
    h, w = gt_binary.shape
    if bg_image is not None:
        overlay = bg_image.copy()
        # Darken background so vessels pop
        overlay = (overlay.astype(np.float32) * 0.3).astype(np.uint8)
    else:
        overlay = np.zeros((h, w, 3), dtype=np.uint8)

    tp = (pred_binary == 1) & (gt_binary == 1)
    fp = (pred_binary == 1) & (gt_binary == 0)
    fn = (pred_binary == 0) & (gt_binary == 1)

    # BGR: Green=(0,255,0), Red=(0,0,255), Blue=(255,0,0)
    overlay[tp] = [0, 255, 0]
    overlay[fp] = [255, 0, 0]
    overlay[fn] = [0, 0, 255]

    return overlay


# ============================================================
# 2. AUTO-FIND BEST ZOOM REGIONS
# ============================================================
def find_best_zoom_regions(gt_binary, pred_binaries, region_size=(48, 30), top_k=2):
    """
    Find regions where predictions progressively improve across ablation models.
    gt_binary: (H,W) uint8 0/1
    pred_binaries: list of (H,W) uint8 0/1, ordered by ablation sequence
    region_size: (width, height) of ROI
    Returns: list of (x, y, w, h)
    """
    rw, rh = region_size
    h, w = gt_binary.shape

    # Progressive gain: pixels recovered from FN of previous model
    gain_map = np.zeros((h, w), dtype=np.float32)
    for i in range(1, len(pred_binaries)):
        prev_fn = gt_binary * (1 - pred_binaries[i - 1])
        curr_tp = gt_binary * pred_binaries[i]
        recovered = prev_fn * curr_tp
        gain_map += recovered

    # Penalize regions where later models regress
    mono_map = np.ones((h, w), dtype=np.float32)
    for i in range(1, len(pred_binaries)):
        regression = (pred_binaries[i - 1] * gt_binary) * (1 - pred_binaries[i])
        mono_map -= regression * 0.5
    mono_map = np.clip(mono_map, 0, 1)

    score_map = gain_map * mono_map

    # Sliding window via convolution
    kernel = np.ones((rh, rw), dtype=np.float32)
    region_scores = fftconvolve(score_map, kernel, mode='valid')

    results = []
    for _ in range(top_k):
        idx = np.unravel_index(np.argmax(region_scores), region_scores.shape)
        y, x = idx
        results.append((x, y, rw, rh))
        # Suppress neighbors
        sy = max(0, y - rh * 2)
        ey = min(region_scores.shape[0], y + rh * 2)
        sx = max(0, x - rw * 2)
        ex = min(region_scores.shape[1], x + rw * 2)
        region_scores[sy:ey, sx:ex] = -1

    return results


# ============================================================
# 3. DRAW ROI BOXES ON IMAGE
# ============================================================
def draw_roi_boxes(image, rois, color=(0, 0, 255), thickness=2):
    """
    image: (H,W,3) BGR
    rois: list of (x, y, w, h)
    Returns: image with rectangles drawn
    """
    img = image.copy()
    for (x, y, w, h) in rois:
        cv2.rectangle(img, (x - 1, y - 1), (x + w + 1, y + h + 1), color, thickness)
    return img


# ============================================================
# 4. EXTRACT & ZOOM ROI
# ============================================================
def extract_zoom_roi(image, roi, zoom_size=(448, 448), border=3, border_color=(0, 0, 255)):
    """
    image: (H,W,3) BGR
    roi: (x, y, w, h)
    Returns: zoomed ROI with colored border
    """
    x, y, w, h = roi
    crop = image[y:y + h, x:x + w]
    zoomed = cv2.resize(crop, zoom_size, interpolation=cv2.INTER_LANCZOS4)
    # Add border
    zoomed = cv2.copyMakeBorder(zoomed, border, border, border, border,
                                cv2.BORDER_CONSTANT, value=border_color)
    return zoomed


# ============================================================
# 5. MAIN PIPELINE
# ============================================================
class AblationVisualizer:
    def __init__(self, model_names, checkpoints_path, output_dir='./output'):
        self.model_names = model_names
        self.checkpoints_path = checkpoints_path
        self.output_dir = output_dir

    def infer_all_models(self, input_path, suffix='drive', ablation=False):
        """
        Run inference for all models. Returns list of (H,W) uint8 0-255 masks.
        >>> REPLACE THIS with your actual inference code <<<
        """
        import torch
        thrs = np.linspace(0.487, 0.488, len(self.model_names))[::-1]
        masks = []
        for i, model_name in enumerate(self.model_names):
            from load_model import load_model_class
            load_model_class(model_name if model_name != 'ours_woLoss' else 'our_net')
            ckpt = f'{model_name}_{suffix}.pt' if suffix else f'{model_name}.pt'
            model = torch.load(
                os.path.join(self.checkpoints_path, ckpt),
                map_location='cuda' if torch.cuda.is_available() else 'cpu',
                weights_only=False
            )
            # Use your existing infer_model logic here
            dis = self.infer_model(model, input_path,preprocessing_img if  model_name != 'our_net' else lambda x: cv2.cvtColor(x, cv2.COLOR_RGB2GRAY),
                                   get_test_patch_transforms_v1 if (ablation and model_name!='our_net') else get_test_patch_transforms,thrs[i])
            masks.append(dis)
        return masks
    def infer_model(self,model, image,preprocessing_func,get_test_patch_transforms=get_test_patch_transforms,thr=0.47):
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = model.to(device)
        model.eval()
        with torch.inference_mode():
            if isinstance(image, str):
                image = np.array(Image.open(image))
            preprocessed_image = preprocessing_func(image)
            img_tensor = get_test_patch_transforms()(image=preprocessed_image)['image'].to(device)
            _, original_h, original_w = img_tensor.shape

            
            img_tensor = mirror_padding_v2(img_tensor).unsqueeze(0)
            B, C, H, W = img_tensor.shape
            num_patch = ((H-64)//32+1, (W-64)//8+1)
            image_patches, tmp_stride = extract_patches_with_target_count(img_tensor, 64, num_patch)
            if len(image_patches.shape) > 4:
                image_patches = image_patches.flatten(0, 1)
            chunk_size = max(image_patches.shape[0] // 128, 1)
            chunk_image = torch.chunk(image_patches, chunk_size, 0)

            out_sample = []
            for c_image in chunk_image:
                with torch.inference_mode():
                    prob = model(c_image)
                out_sample.append(prob)
            prob = torch.cat(out_sample, 0)
            prob = prob.view(B, -1, 1, 64, 64)
            prob = reverse_to_original_image(prob, (H, W), 64, tmp_stride).squeeze()[:original_h, :original_w]

            # Threshold => binary mask (numpy)
            pred_mask = (prob >=thr).to(torch.uint8).detach().cpu().numpy()  # shape (h,w), 0/1
            display_mask = (pred_mask * 255).astype(np.uint8)
            return display_mask
    # return infer_model(model,image_path,)
        """Wrapper - plug in your existing infer_model code"""
        # ... your existing inference code ...
        # Return (H,W) uint8 0-255 mask
        
        raise NotImplementedError("Plug in your infer_model here")

    def run(self, input_path, gt_path, suffix='drive',
            pos_0=None, pos_1=None,
            region_size=(48, 30), zoom_size=(448, 448),
            ablation=False):
        """
        Full pipeline:
        1. Infer all models
        2. Auto-find 2 best ROI (or use provided pos_0, pos_1)
        3. For each model: save overlay + 2 zoomed ROIs
        """
        img_name = os.path.splitext(os.path.basename(input_path))[0]
        out_root = os.path.join(self.output_dir, img_name)
        os.makedirs(out_root, exist_ok=True)

        # Load GT & original
        gt_raw = cv2.imread(gt_path, 0)
        gt_bin = np.ceil(gt_raw.astype(np.float32) / 255).astype(np.uint8)
        orig_bgr = cv2.imread(input_path, 1)

        # Infer
        pred_masks_255 = self.infer_all_models(input_path, suffix, ablation)
        pred_bins = [np.ceil(m.astype(np.float32) / 255).astype(np.uint8) for m in pred_masks_255]

        # Auto-find ROIs if not provided
        if pos_0 is None or pos_1 is None:
            rois = find_best_zoom_regions(gt_bin, pred_bins, region_size, top_k=2)
            pos_0, pos_1 = rois[0], rois[1]
            print(f"[{img_name}] Auto ROI: pos_0={pos_0}, pos_1={pos_1}")

        rois = [pos_0, pos_1]

        # --- Save original image with ROI boxes ---
        orig_with_boxes = draw_roi_boxes(orig_bgr, rois, color=(0, 0, 255), thickness=2)
        cv2.imwrite(os.path.join(out_root, 'original_with_boxes.png'), orig_with_boxes)

        # --- Save original zoomed ROIs ---
        for ri, roi in enumerate(rois):
            z = extract_zoom_roi(orig_bgr, roi, zoom_size)
            cv2.imwrite(os.path.join(out_root, f'original_zoom_{ri}.png'), z)

        # --- GT overlay (all green = TP concept) ---
        gt_overlay = create_tp_fp_fn_overlay(gt_bin, gt_bin, orig_bgr)
        gt_with_boxes = draw_roi_boxes(gt_overlay, rois, color=(0, 0, 255), thickness=2)
        cv2.imwrite(os.path.join(out_root, 'gt_overlay.png'), gt_with_boxes)
        for ri, roi in enumerate(rois):
            z = extract_zoom_roi(gt_overlay, roi, zoom_size)
            cv2.imwrite(os.path.join(out_root, f'gt_zoom_{ri}.png'), z)

        # --- Each model ---
        for mi, model_name in enumerate(self.model_names):
            overlay = create_tp_fp_fn_overlay(pred_bins[mi], gt_bin, orig_bgr)
            overlay_with_boxes = draw_roi_boxes(overlay, rois, color=(0, 0, 255), thickness=2)
            cv2.imwrite(os.path.join(out_root, f'{model_name}_overlay.png'), overlay_with_boxes)

            for ri, roi in enumerate(rois):
                z = extract_zoom_roi(overlay, roi, zoom_size)
                cv2.imwrite(os.path.join(out_root, f'{model_name}_zoom_{ri}.png'), z)

        print(f"[{img_name}] Saved to {out_root}")
        print(f"  Files per model: overlay + 2 zooms")
        print(f"  ROI positions: {rois}")
        def concat_zoom_row(out_root, model_names, roi_idx, output_name=None):
            """
            Concat: original_zoom_{roi_idx} + gt_zoom_{roi_idx} + each model's zoom_{roi_idx}
            All horizontally, saved as one image.
            """
            order = ['original', 'gt'] + model_names
            imgs = []
            for name in order:
                path = os.path.join(out_root, f'{name}_zoom_{roi_idx}.png')
                img = cv2.imread(path)
                if img is not None:
                    imgs.append(img)

            if not imgs:
                return

            # Resize all to same height
            target_h = imgs[0].shape[0]
            resized = []
            for img in imgs:
                if img.shape[0] != target_h:
                    scale = target_h / img.shape[0]
                    img = cv2.resize(img, (int(img.shape[1] * scale), target_h))
                resized.append(img)

            cat = np.concatenate(resized, axis=1)
            out_name = output_name or f'cat_zoom_{roi_idx}.png'
            cv2.imwrite(os.path.join(out_root, out_name), cat)
            print(f"  Saved {out_name} ({cat.shape[1]}x{cat.shape[0]})")

        for ri in range(len(rois)):
            concat_zoom_row(out_root, self.model_names, ri)
        overlay_imgs = []
        for name in ['original_with_boxes', 'gt_overlay'] + [f'{m}_overlay' for m in self.model_names]:
            p = os.path.join(out_root, f'{name}.png')
            img = cv2.imread(p)
            if img is not None:
                overlay_imgs.append(img)
        if overlay_imgs:
            target_h = overlay_imgs[0].shape[0]
            overlay_imgs = [cv2.resize(im, (int(im.shape[1] * target_h / im.shape[0]), target_h))
                           if im.shape[0] != target_h else im for im in overlay_imgs]
            cat_overlay = np.concatenate(overlay_imgs, axis=1)
            cv2.imwrite(os.path.join(out_root, 'cat_overlays.png'), cat_overlay)
            print(f"  Saved cat_overlays.png")
        return rois, pred_bins, gt_bin


# ============================================================
# 6. USAGE
# ============================================================
if __name__ == '__main__':
    model_names = [
        'baseline',
        'baseline_RHMA_woMDSA',
        'baseline_RHMA_SAG_woMDSA',
        'baseline_RHMA_SAG_MDSA',
        'ours_woLoss',
        'our_net'
    ]

    viz = AblationVisualizer(
        model_names=model_names,
        checkpoints_path='/workspace/Retinal-vessels-segmentation/checkpoints',
        output_dir='./output'
    )

    img_paths = sorted(glob.glob('/workspace/Retinal-vessels-segmentation/data/DRIVE/*/images/*'))
    gt_paths = sorted(glob.glob('/workspace/Retinal-vessels-segmentation/data/DRIVE/*/mask/*'))

    # Pass 1: find best image + ROIs
    best_score = -1
    best_idx = 0
    best_rois = None

    # for i in range(len(img_paths)):
    #     print(f"\n=== Image {i}: {img_paths[i]} ===")
    #     try:
    #         rois = viz.run(img_paths[i], gt_paths[i], suffix='', ablation=True)
    #     except NotImplementedError:
    #         # If inference not plugged in, skip
    #         print("  [SKIP] Plug in your infer_model first")
    #         break
    all_scores = []
    for i in range(len(img_paths)):
        print(f"\n=== Image {i}: {img_paths[i]} ===")
        rois, pred_bins, gt_bin = viz.run(img_paths[i], gt_paths[i], suffix='', ablation=True)
        score, details = score_image_for_visualization(gt_bin, pred_bins)
        all_scores.append((i, score, details, rois))
        print(f"  Score={score:.1f} | mono={details['mono_ratio']:.0%} | even={details['evenness']:.2f} | gains={details['step_gains']}")

    # Rank
    all_scores.sort(key=lambda x: x[1], reverse=True)
    print("\n=== TOP 5 IMAGES ===")
    for rank, (idx, score, details, rois) in enumerate(all_scores[:5]):
        print(f"  #{rank+1}: image {idx} ({img_paths[idx]}) score={score:.1f} rois={rois}")
    

In [ ]:
"""
Ablation Study Visualization v2
- Overlay: Green=TP, Blue=FP, Red=FN
- Pred mask: binary (white on black) + overlay version
- Auto-find 1 ROI region where our_net is BEST
- Yellow border highlight for our_net column
- Layout per image:
    Left: original full image with ROI box
    Row 1: zoomed overlay   (original | GT | model1 | ... | our_net[yellow])
    Row 2: zoomed pred_mask (original | GT | model1 | ... | our_net[yellow])
- Concatenated output for quick review
"""

import os
import cv2
import glob
import numpy as np
from PIL import Image
from scipy.signal import fftconvolve


# ============================================================
# 1. COLOR OVERLAY (TP=Green, FP=Blue, FN=Red)
# ============================================================
def create_tp_fp_fn_overlay(pred_binary, gt_binary, bg_image=None):
    """
    pred_binary, gt_binary: (H,W) uint8, values 0 or 1
    bg_image: optional (H,W,3) BGR background; if None, use black
    Returns: (H,W,3) BGR overlay image
    """
    h, w = gt_binary.shape
    if bg_image is not None:
        overlay = bg_image.copy()
        overlay = (overlay.astype(np.float32) * 0.3).astype(np.uint8)
    else:
        overlay = np.zeros((h, w, 3), dtype=np.uint8)

    tp = (pred_binary == 1) & (gt_binary == 1)
    fp = (pred_binary == 1) & (gt_binary == 0)
    fn = (pred_binary == 0) & (gt_binary == 1)

    # BGR: Green=(0,255,0), Blue=(255,0,0), Red=(0,0,255)
    overlay[tp] = [0, 255, 0]
    overlay[fp] = [255, 0, 0]
    overlay[fn] = [0, 0, 255]

    return overlay


# ============================================================
# 2. CREATE PRED MASK VISUALIZATION (white vessels on black)
# ============================================================
def create_pred_mask_vis(pred_binary):
    """pred_binary: (H,W) uint8 0/1 => (H,W,3) BGR white on black"""
    vis = np.zeros((*pred_binary.shape, 3), dtype=np.uint8)
    vis[pred_binary == 1] = [255, 255, 255]
    return vis


def create_gt_mask_vis(gt_binary):
    """Same as pred but for ground truth"""
    return create_pred_mask_vis(gt_binary)


# ============================================================
# 3. FIND BEST ROI WHERE our_net IS BEST (1 region)
# ============================================================
def find_best_ournet_region(gt_binary, pred_binaries, model_names,
                            ournet_name='our_net', region_size=(80, 50),
                            top_k=1):
    """
    Find region where our_net has highest F1 advantage over other models.
    Returns: list of (x, y, w, h)
    """
    rw, rh = region_size
    h, w = gt_binary.shape

    # Find our_net index
    ournet_idx = None
    for i, name in enumerate(model_names):
        if name == ournet_name:
            ournet_idx = i
            break
    if ournet_idx is None:
        raise ValueError(f"'{ournet_name}' not found in model_names")

    ournet_pred = pred_binaries[ournet_idx]

    # Build advantage map: pixels where our_net is TP but others are FN
    advantage_map = np.zeros((h, w), dtype=np.float32)
    ournet_tp = (ournet_pred == 1) & (gt_binary == 1)

    for i, pred in enumerate(pred_binaries):
        if i == ournet_idx:
            continue
        other_fn = (pred == 0) & (gt_binary == 1)
        # Pixels our_net recovers but other model misses
        advantage_map += (ournet_tp & other_fn).astype(np.float32)

    # Also penalize our_net's own FP to prefer clean regions
    ournet_fp = (ournet_pred == 1) & (gt_binary == 0)
    penalty_map = ournet_fp.astype(np.float32) * 0.3

    score_map = advantage_map - penalty_map
    score_map = np.clip(score_map, 0, None)

    # Sliding window via convolution
    kernel = np.ones((rh, rw), dtype=np.float32)
    region_scores = fftconvolve(score_map, kernel, mode='valid')

    results = []
    for _ in range(top_k):
        idx = np.unravel_index(np.argmax(region_scores), region_scores.shape)
        y, x = idx
        results.append((x, y, rw, rh))
        # Suppress neighbors
        sy = max(0, y - rh * 3)
        ey = min(region_scores.shape[0], y + rh * 3)
        sx = max(0, x - rw * 3)
        ex = min(region_scores.shape[1], x + rw * 3)
        region_scores[sy:ey, sx:ex] = -1

    return results


# ============================================================
# 4. DRAW ROI BOX ON FULL IMAGE
# ============================================================
def draw_roi_boxes(image, rois, color=(0, 0, 255), thickness=2):
    img = image.copy()
    for (x, y, w, h) in rois:
        cv2.rectangle(img, (x - 1, y - 1), (x + w + 1, y + h + 1), color, thickness)
    return img


def draw_connecting_lines(full_img, roi, zoom_pos, zoom_size, color=(0, 0, 255), thickness=1):
    """Draw lines connecting ROI box corners to zoom panel corners."""
    img = full_img.copy()
    x, y, w, h = roi
    zx, zy = zoom_pos
    zw, zh = zoom_size

    # Top-left of ROI to top-left of zoom
    cv2.line(img, (x, y), (zx, zy), color, thickness, cv2.LINE_AA)
    # Bottom-left of ROI to bottom-left of zoom
    cv2.line(img, (x, y + h), (zx, zy + zh), color, thickness, cv2.LINE_AA)

    return img


# ============================================================
# 5. EXTRACT & ZOOM ROI
# ============================================================
def extract_zoom_roi(image, roi, zoom_size=(448, 448),
                     border=3, border_color=(0, 0, 255)):
    x, y, w, h = roi
    crop = image[y:y + h, x:x + w]
    zoomed = cv2.resize(crop, zoom_size, interpolation=cv2.INTER_LANCZOS4)
    zoomed = cv2.copyMakeBorder(zoomed, border, border, border, border,
                                cv2.BORDER_CONSTANT, value=border_color)
    return zoomed


def extract_zoom_roi_yellow(image, roi, zoom_size=(448, 448),
                            border=5, border_color=(0, 255, 255)):
    """Same but with thick yellow border for our_net highlight."""
    return extract_zoom_roi(image, roi, zoom_size, border, border_color)


# ============================================================
# 6. SCORE IMAGE FOR QUICK REVIEW
# ============================================================
def score_image_for_visualization(gt_bin, pred_bins, model_names, ournet_name='our_net'):
    """
    Score how good an image is for showing our_net advantage.
    Higher = our_net has more advantage in this image.
    """
    ournet_idx = model_names.index(ournet_name)
    ournet_pred = pred_bins[ournet_idx]

    # our_net's F1-like score
    ournet_tp = ((ournet_pred == 1) & (gt_bin == 1)).sum()
    ournet_fp = ((ournet_pred == 1) & (gt_bin == 0)).sum()
    ournet_fn = ((ournet_pred == 0) & (gt_bin == 1)).sum()
    ournet_f1 = 2 * ournet_tp / (2 * ournet_tp + ournet_fp + ournet_fn + 1e-8)

    # Average F1 of other models
    other_f1s = []
    for i, pred in enumerate(pred_bins):
        if i == ournet_idx:
            continue
        tp = ((pred == 1) & (gt_bin == 1)).sum()
        fp = ((pred == 1) & (gt_bin == 0)).sum()
        fn = ((pred == 0) & (gt_bin == 1)).sum()
        f1 = 2 * tp / (2 * tp + fp + fn + 1e-8)
        other_f1s.append(f1)

    avg_other_f1 = np.mean(other_f1s) if other_f1s else 0

    # Advantage = how much better our_net is
    advantage = ournet_f1 - avg_other_f1

    # Also count pixels uniquely recovered by our_net
    unique_recovery = 0
    for i, pred in enumerate(pred_bins):
        if i == ournet_idx:
            continue
        other_fn = (pred == 0) & (gt_bin == 1)
        ournet_tp_mask = (ournet_pred == 1) & (gt_bin == 1)
        unique_recovery += (other_fn & ournet_tp_mask).sum()

    score = advantage * 1000 + unique_recovery * 0.1

    details = {
        'ournet_f1': ournet_f1,
        'avg_other_f1': avg_other_f1,
        'advantage': advantage,
        'unique_recovery': int(unique_recovery),
    }
    return score, details


# ============================================================
# 7. MAIN PIPELINE
# ============================================================
class AblationVisualizer:
    def __init__(self, model_names, checkpoints_path, output_dir='./output',
                 ournet_name='our_net'):
        self.model_names = model_names
        self.checkpoints_path = checkpoints_path
        self.output_dir = output_dir
        self.ournet_name = ournet_name

    def infer_all_models(self, input_path, suffix='drive', ablation=False):
        """
        Run inference for all models. Returns list of (H,W) uint8 0-255 masks.
        >>> REPLACE THIS with your actual inference code <<<
        """
        import torch
        thrs = np.linspace(0.487, 0.4871, len(self.model_names))[::-1]
        masks = []
        for i, model_name in enumerate(self.model_names):
            from load_model import load_model_class
            load_model_class(model_name if model_name != 'ours_woLoss' else 'our_net')
            ckpt = f'{model_name}_{suffix}.pt' if suffix else f'{model_name}.pt'
            model = torch.load(
                os.path.join(self.checkpoints_path, ckpt),
                map_location='cuda' if torch.cuda.is_available() else 'cpu',
                weights_only=False
            )
            dis = self.infer_model(
                model, input_path,
                preprocessing_img if model_name != 'our_net' else lambda x: cv2.cvtColor(x, cv2.COLOR_RGB2GRAY),
                get_test_patch_transforms if (ablation and model_name != 'our_net') else get_test_patch_transforms,
                thrs[i]
            )
            masks.append(dis)
        return masks

    def infer_model(self, model, image, preprocessing_func,
                    get_test_patch_transforms_func=None, thr=0.47):
        import torch
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = model.to(device)
        model.eval()
        with torch.inference_mode():
            if isinstance(image, str):
                image = np.array(Image.open(image))
            preprocessed_image = preprocessing_func(image)
            img_tensor = get_test_patch_transforms_func()(image=preprocessed_image)['image'].to(device)
            _, original_h, original_w = img_tensor.shape

            img_tensor = mirror_padding_v2(img_tensor).unsqueeze(0)
            B, C, H, W = img_tensor.shape
            num_patch = ((H - 64) // 32 + 1, (W - 64) // 8 + 1)
            image_patches, tmp_stride = extract_patches_with_target_count(img_tensor, 64, num_patch)
            if len(image_patches.shape) > 4:
                image_patches = image_patches.flatten(0, 1)
            chunk_size = max(image_patches.shape[0] // 128, 1)
            chunk_image = torch.chunk(image_patches, chunk_size, 0)

            out_sample = []
            for c_image in chunk_image:
                with torch.inference_mode():
                    prob = model(c_image)
                out_sample.append(prob)
            prob = torch.cat(out_sample, 0)
            prob = prob.view(B, -1, 1, 64, 64)
            prob = reverse_to_original_image(prob, (H, W), 64, tmp_stride).squeeze()[:original_h, :original_w]

            pred_mask = (prob >= thr).to(torch.uint8).detach().cpu().numpy()
            display_mask = (pred_mask * 255).astype(np.uint8)
            return display_mask

    def run(self, input_path, gt_path, suffix='drive',
            pos_0=None, region_size=(80, 50), zoom_size=(448, 448),
            ablation=False):
        """
        Full pipeline:
        1. Infer all models
        2. Find 1 ROI where our_net is best (or use provided pos_0)
        3. For each model: save overlay zoom + pred_mask zoom
        4. Concatenate rows for quick review

        Output structure per image:
          original_with_box.png
          original_zoom.png
          gt_overlay_zoom.png, gt_mask_zoom.png, gt_mask_with_box.png
          {model}_overlay_zoom.png, {model}_mask_zoom.png, {model}_mask_with_box.png
          cat_overlay.png        (row of overlay zooms)
          cat_mask.png           (row of pred_mask zooms)
          cat_mask_with_box.png  (row of full-size masks with ROI box)
          cat_combined.png       (all 3 rows stacked)
        """
        img_name = os.path.splitext(os.path.basename(input_path))[0]
        out_root = os.path.join(self.output_dir, img_name)
        os.makedirs(out_root, exist_ok=True)

        # Load GT & original
        gt_raw = cv2.imread(gt_path, 0)
        gt_bin = np.ceil(gt_raw.astype(np.float32) / 255).astype(np.uint8)
        orig_bgr = cv2.imread(input_path, 1)

        # Infer
        pred_masks_255 = self.infer_all_models(input_path, suffix, ablation)
        pred_bins = [np.ceil(m.astype(np.float32) / 255).astype(np.uint8)
                     for m in pred_masks_255]

        # Find best ROI for our_net
        if pos_0 is None:
            rois = find_best_ournet_region(
                gt_bin, pred_bins, self.model_names,
                ournet_name=self.ournet_name,
                region_size=region_size, top_k=1
            )
            pos_0 = rois[0]
            print(f"[{img_name}] Auto ROI (our_net best): pos_0={pos_0}")

        roi = pos_0
        rois = [roi]

        # Full-size output dimensions (H=999, W=960)
        full_size = (960, 999)  # (w, h) for cv2.resize

        # ---- Full image with ROI box (yellow for emphasis) ----
        orig_with_box = draw_roi_boxes(orig_bgr, rois,
                                       color=(0, 255, 255), thickness=2)
        orig_with_box = cv2.resize(orig_with_box, full_size, interpolation=cv2.INTER_LANCZOS4)
        cv2.imwrite(os.path.join(out_root, 'original_with_box.png'), orig_with_box)

        # ---- Original zoomed ROI ----
        orig_zoom = extract_zoom_roi(orig_bgr, roi, zoom_size,
                                     border=3, border_color=(0, 255, 255))
        cv2.imwrite(os.path.join(out_root, 'original_zoom.png'), orig_zoom)

        # ---- GT overlay zoom ----
        gt_overlay = create_tp_fp_fn_overlay(gt_bin, gt_bin, orig_bgr)
        gt_overlay_zoom = extract_zoom_roi(gt_overlay, roi, zoom_size,
                                           border=3, border_color=(0, 255, 255))
        cv2.imwrite(os.path.join(out_root, 'gt_overlay_zoom.png'), gt_overlay_zoom)

        # ---- GT mask zoom ----
        gt_mask_vis = create_gt_mask_vis(gt_bin)
        gt_mask_zoom = extract_zoom_roi(gt_mask_vis, roi, zoom_size,
                                        border=3, border_color=(0, 255, 255))
        cv2.imwrite(os.path.join(out_root, 'gt_mask_zoom.png'), gt_mask_zoom)

        # ---- GT mask with box (full size) ----
        gt_mask_with_box = draw_roi_boxes(gt_mask_vis, rois,
                                          color=(0, 255, 255), thickness=2)
        gt_mask_with_box = cv2.resize(gt_mask_with_box, full_size, interpolation=cv2.INTER_LANCZOS4)
        cv2.imwrite(os.path.join(out_root, 'gt_mask_with_box.png'), gt_mask_with_box)

        # ---- Each model: overlay + pred_mask + mask_with_box ----
        for mi, model_name in enumerate(self.model_names):
            is_ournet = (model_name == self.ournet_name)

            # Overlay
            overlay = create_tp_fp_fn_overlay(pred_bins[mi], gt_bin, orig_bgr)
            if is_ournet:
                overlay_zoom = extract_zoom_roi_yellow(overlay, roi, zoom_size,
                                                       border=5,
                                                       border_color=(0, 255, 255))
            else:
                overlay_zoom = extract_zoom_roi(overlay, roi, zoom_size,
                                                border=3,
                                                border_color=(0, 255, 255))
            cv2.imwrite(os.path.join(out_root, f'{model_name}_overlay_zoom.png'),
                        overlay_zoom)

            # Pred mask (binary white on black)
            mask_vis = create_pred_mask_vis(pred_bins[mi])
            if is_ournet:
                mask_zoom = extract_zoom_roi_yellow(mask_vis, roi, zoom_size,
                                                     border=5,
                                                     border_color=(0, 255, 255))
            else:
                mask_zoom = extract_zoom_roi(mask_vis, roi, zoom_size,
                                             border=3,
                                             border_color=(0, 255, 255))
            cv2.imwrite(os.path.join(out_root, f'{model_name}_mask_zoom.png'),
                        mask_zoom)

            # Pred mask with ROI box (full size)
            mask_with_box = draw_roi_boxes(mask_vis, rois,
                                           color=(0, 255, 255) if is_ournet else (0, 0, 255),
                                           thickness=2)
            mask_with_box = cv2.resize(mask_with_box, full_size, interpolation=cv2.INTER_LANCZOS4)
            cv2.imwrite(os.path.join(out_root, f'{model_name}_mask_with_box.png'),
                        mask_with_box)

        # ---- Concatenate rows ----
        def load_and_resize(path, target_h):
            img = cv2.imread(path)
            if img is None:
                return None
            if img.shape[0] != target_h:
                scale = target_h / img.shape[0]
                img = cv2.resize(img, (int(img.shape[1] * scale), target_h))
            return img

        # Column order: original | GT | model1 | model2 | ... | our_net
        col_order = ['original', 'gt'] + self.model_names

        # Row 1: Overlay zooms
        overlay_imgs = []
        for name in col_order:
            if name == 'original':
                path = os.path.join(out_root, 'original_zoom.png')
            elif name == 'gt':
                path = os.path.join(out_root, 'gt_overlay_zoom.png')
            else:
                path = os.path.join(out_root, f'{name}_overlay_zoom.png')
            img = cv2.imread(path)
            if img is not None:
                overlay_imgs.append(img)

        # Row 2: Pred mask zooms
        mask_imgs = []
        for name in col_order:
            if name == 'original':
                # For mask row, show original zoom or skip
                # Show original as grayscale-ish reference
                path = os.path.join(out_root, 'original_zoom.png')
            elif name == 'gt':
                path = os.path.join(out_root, 'gt_mask_zoom.png')
            else:
                path = os.path.join(out_root, f'{name}_mask_zoom.png')
            img = cv2.imread(path)
            if img is not None:
                mask_imgs.append(img)

        if overlay_imgs:
            target_h = overlay_imgs[0].shape[0]
            overlay_row = [load_and_resize(
                os.path.join(out_root,
                             ('original_zoom.png' if n == 'original' else
                              'gt_overlay_zoom.png' if n == 'gt' else
                              f'{n}_overlay_zoom.png')),
                target_h
            ) for n in col_order]
            overlay_row = [x for x in overlay_row if x is not None]
            if overlay_row:
                cat_overlay = np.concatenate(overlay_row, axis=1)
                cv2.imwrite(os.path.join(out_root, 'cat_overlay.png'), cat_overlay)

            mask_row = [load_and_resize(
                os.path.join(out_root,
                             ('original_zoom.png' if n == 'original' else
                              'gt_mask_zoom.png' if n == 'gt' else
                              f'{n}_mask_zoom.png')),
                target_h
            ) for n in col_order]
            mask_row = [x for x in mask_row if x is not None]
            if mask_row:
                cat_mask = np.concatenate(mask_row, axis=1)
                cv2.imwrite(os.path.join(out_root, 'cat_mask.png'), cat_mask)

            # Row 3: Full-size pred masks with ROI box
            mask_box_row = [load_and_resize(
                os.path.join(out_root,
                             ('original_with_box.png' if n == 'original' else
                              'gt_mask_with_box.png' if n == 'gt' else
                              f'{n}_mask_with_box.png')),
                target_h
            ) for n in col_order]
            mask_box_row = [x for x in mask_box_row if x is not None]
            if mask_box_row:
                cat_mask_box = np.concatenate(mask_box_row, axis=1)
                cv2.imwrite(os.path.join(out_root, 'cat_mask_with_box.png'), cat_mask_box)

            # Combined: overlay zoom | mask zoom | mask_with_box
            all_rows = []
            for row_imgs in [overlay_row, mask_row, mask_box_row]:
                if row_imgs:
                    all_rows.append(np.concatenate(row_imgs, axis=1))
            if len(all_rows) > 1:
                min_w = min(r.shape[1] for r in all_rows)
                all_rows = [r[:, :min_w] for r in all_rows]
                cat_combined = np.concatenate(all_rows, axis=0)
                cv2.imwrite(os.path.join(out_root, 'cat_combined.png'), cat_combined)

        # ---- Full-image panel ----
        orig_resized = cv2.resize(orig_with_box, full_size, interpolation=cv2.INTER_LANCZOS4)
        cv2.imwrite(os.path.join(out_root, 'panel_original.png'), orig_resized)

        print(f"[{img_name}] Saved to {out_root}")
        print(f"  Files: original_with_box, original_zoom, gt_overlay/mask_zoom/mask_with_box,")
        print(f"         {len(self.model_names)} models x (overlay_zoom + mask_zoom + mask_with_box),")
        print(f"         cat_overlay, cat_mask, cat_mask_with_box, cat_combined")
        print(f"  ROI: {roi}")

        return roi, pred_bins, gt_bin

    def run_all_and_rank(self, img_paths, gt_paths, suffix='',
                         region_size=(80,60), zoom_size=(448, 448),
                         ablation=False):
        """
        Run all images, score them, rank by our_net advantage.
        """
        all_scores = []
        for i in range(len(img_paths)):
            print(f"\n=== Image {i}: {img_paths[i]} ===")
            roi, pred_bins, gt_bin = self.run(
                img_paths[i], gt_paths[i],
                suffix=suffix,
                region_size=region_size,
                zoom_size=zoom_size,
                ablation=ablation
            )
            score, details = score_image_for_visualization(
                gt_bin, pred_bins, self.model_names, self.ournet_name
            )
            all_scores.append((i, score, details, roi))
            print(f"  Score={score:.1f} | our_f1={details['ournet_f1']:.4f} "
                  f"| avg_other={details['avg_other_f1']:.4f} "
                  f"| advantage={details['advantage']:.4f} "
                  f"| unique_recovery={details['unique_recovery']}")

        # Rank
        all_scores.sort(key=lambda x: x[1], reverse=True)
        print("\n" + "=" * 60)
        print("TOP 5 IMAGES (best our_net advantage)")
        print("=" * 60)
        for rank, (idx, score, details, roi) in enumerate(all_scores[:5]):
            print(f"  #{rank + 1}: image {idx} "
                  f"({os.path.basename(img_paths[idx])}) "
                  f"score={score:.1f} roi={roi}")
            print(f"       our_f1={details['ournet_f1']:.4f} "
                  f"avg_other={details['avg_other_f1']:.4f} "
                  f"adv={details['advantage']:.4f}")

        # Build mega-cat of top 5 for quick review
        self._build_mega_cat(img_paths, all_scores[:5])

        return all_scores

    def _build_mega_cat(self, img_paths, top_scores):
        """
        Stack cat_combined of top images vertically for one-glance review.
        Add image index label on the left.
        """
        rows = []
        for idx, score, details, roi in top_scores:
            img_name = os.path.splitext(os.path.basename(img_paths[idx]))[0]
            cat_path = os.path.join(self.output_dir, img_name, 'cat_combined.png')
            if not os.path.exists(cat_path):
                continue
            cat_img = cv2.imread(cat_path)
            if cat_img is None:
                continue

            # Add label on the left
            label = f"#{idx} s={score:.0f}"
            label_w = 120
            label_panel = np.zeros((cat_img.shape[0], label_w, 3), dtype=np.uint8)
            # Put text
            font = cv2.FONT_HERSHEY_SIMPLEX
            cv2.putText(label_panel, f"#{idx}",
                        (5, cat_img.shape[0] // 2 - 10),
                        font, 0.6, (255, 255, 255), 1, cv2.LINE_AA)
            cv2.putText(label_panel, f"s={score:.0f}",
                        (5, cat_img.shape[0] // 2 + 20),
                        font, 0.5, (0, 255, 255), 1, cv2.LINE_AA)

            row = np.concatenate([label_panel, cat_img], axis=1)
            rows.append(row)

        if rows:
            # Match widths
            max_w = max(r.shape[1] for r in rows)
            padded = []
            for r in rows:
                if r.shape[1] < max_w:
                    pad = np.zeros((r.shape[0], max_w - r.shape[1], 3), dtype=np.uint8)
                    r = np.concatenate([r, pad], axis=1)
                padded.append(r)

            # Add thin separator between images
            sep_h = 4
            final_rows = []
            for i, r in enumerate(padded):
                final_rows.append(r)
                if i < len(padded) - 1:
                    sep = np.ones((sep_h, max_w, 3), dtype=np.uint8) * 80
                    final_rows.append(sep)

            mega = np.concatenate(final_rows, axis=0)
            out_path = os.path.join(self.output_dir, 'mega_review.png')
            cv2.imwrite(out_path, mega)
            print(f"\nSaved mega review: {out_path} ({mega.shape[1]}x{mega.shape[0]})")


# ============================================================
# 8. USAGE
# ============================================================
if __name__ == '__main__':
    model_names = [
        'unet',
        'fr_net',
        'gtdla',
        'edae_net',
        'dysta_net',
        'sfit_net',
        'our_net',
    ]

    viz = AblationVisualizer(
        model_names=model_names,
        checkpoints_path='/workspace/Retinal-vessels-segmentation/checkpoints',
        output_dir='./output',
        ournet_name='our_net',
    )

    img_paths = sorted(glob.glob(
        '/workspace/Retinal-vessels-segmentation/data/STARE_F1/*/images/*'))
    gt_paths = sorted(glob.glob(
        '/workspace/Retinal-vessels-segmentation/data/STARE_F1/*/mask/*'))

    # Run all & rank by our_net advantage
    all_scores = viz.run_all_and_rank(
        img_paths, gt_paths,
        suffix='',
        region_size=(80, 60),   # bigger ROI region
        zoom_size=(448, 448),
        ablation=True,
    )